[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Authentication &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, its credentials in the
environment, and `KEY`, `CLIENT_ID` and `CLIENT_SECRET` read from there. Run it first.


In [1]:
import base64
import importlib
import json
import os
import sys
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
os.environ.update(practice_api.credentials())    # where a program finds its credentials
KEY = os.environ["PRACTICE_API_KEY"]
CLIENT_ID = os.environ["PRACTICE_CLIENT_ID"]
CLIENT_SECRET = os.environ["PRACTICE_CLIENT_SECRET"]

print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** The key in a header of the API's own.


In [2]:
body = requests.get(f"{BASE}/me", headers={"X-API-Key": KEY}, timeout=10).json()

print(body["client"], body["scopes"])


station-report ['stations:read']


The practice API found the same key in `X-API-Key` as in `Authorization`, and knew it as
`station-report`, with `stations:read` alone.


**2.** The scope a `403` asks for.


In [3]:
response = requests.get(f"{BASE}/network/maintenance", headers={"Authorization": f"Bearer {KEY}"}, timeout=10)
scheme, _, rest = response.headers["WWW-Authenticate"].partition(" ")
parameters = dict(part.strip().split("=", 1) for part in rest.split(","))

print(response.status_code, parameters["scope"].strip('"'))


403 maintenance:read


After the scheme, a challenge is a list of `name=value` pairs separated by commas, most values in
quotation marks. Splitting on commas works for this header. A value with a comma inside its
quotation marks, as an `error_description` can have, needs a parser that respects the quotation
marks.


**3.** A token's lifetime and scopes.


In [4]:
response = requests.post(f"{BASE}/auth/token", auth=(CLIENT_ID, CLIENT_SECRET),
                         data={"grant_type": "client_credentials"}, timeout=10)
grant = response.json()

print(grant["expires_in"] // 60, "minutes")
print(grant["scope"].split())


60 minutes
['stations:read', 'maintenance:read']


`expires_in` counts seconds, so an hour is `60` minutes, and `scope` is one string of names separated
by spaces, which `split` turns into a list. The token was read into `grant` and never printed.


**4.** The claims of an expired token.


In [5]:
expired = requests.get(f"{BASE}/auth/expired-token", timeout=10).json()["access_token"]
payload = expired.split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))

print(claims["sub"], datetime.fromtimestamp(claims["exp"], tz=timezone.utc))


maintenance-console 2026-02-28 09:00:00+00:00


The token expired at 09:00 UTC on February 28, 2026, a day before the practice API's clock. Its
claims tell that to anyone who holds it, and the practice API reaches the same verdict by comparing
`exp` with its clock.


**5.** A key kept out of what is printed.


In [6]:
def redact(text):
    """text with the key and the client secret replaced by the names of their variables."""
    for name in ["PRACTICE_API_KEY", "PRACTICE_CLIENT_SECRET"]:
        text = text.replace(os.environ[name], f"<{name}>")
    return text


response = requests.get(f"{BASE}/me", params={"api_key": KEY}, timeout=10)

print(redact(response.url))
print(redact(practice_api.access_log()[-1]))


http://127.0.0.1:8765/me?api_key=<PRACTICE_API_KEY>
127.0.0.1 - - [01/Mar/2026 09:00:00] "GET /me?api_key=<PRACTICE_API_KEY> HTTP/1.1" 200 -


`response.url` is the URL requests sent, query and all, and the last line of the access log is the
one the practice API wrote for that request. `redact` looks the values up in the environment, so it
holds no credential itself.


**6.** A token renewed when refused.


In [7]:
def new_token():
    """A new access token for the maintenance console."""
    response = requests.post(f"{BASE}/auth/token", auth=(CLIENT_ID, CLIENT_SECRET),
                             data={"grant_type": "client_credentials"}, timeout=10)
    response.raise_for_status()
    return response.json()["access_token"]


def get_maintenance(token):
    """The maintenance visits, and the token used to get them, renewed if the practice API refused it."""
    url = f"{BASE}/network/maintenance"
    response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=10)
    if response.status_code == 401 and "invalid_token" in response.headers.get("WWW-Authenticate", ""):
        token = new_token()
        response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=10)
    response.raise_for_status()
    return response.json()["visits"], token


expired = requests.get(f"{BASE}/auth/expired-token", timeout=10).json()["access_token"]
visits, used = get_maintenance(expired)

print(len(visits), "visits | a new token:", used != expired)


3 visits | a new token: True


The expired token got a `401` with `invalid_token`, so `get_maintenance` fetched a new token and
asked again. Returning the token lets the caller pass it to the next call, instead of fetching a new
token for every request.


---

&#8592; **Back to:** [Authentication](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/09-authentication.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
